# QR Code & Barcode Scanner demo

This notebook installs the open-source Python package
[`qrcode-barcode-scanner`](https://pypi.org/project/qrcode-barcode-scanner/0.11.0/) 0.11.0 from PyPI and reads a QR code and a Code 128 barcode
from a sample image with it. The engine is written in Rust and runs inside this notebook's
session: the images are decoded on the machine that runs the session, here a Kaggle machine, and
never on the reader's computer. The scanner sends nothing anywhere; the notebook only downloads
the package and the sample image.

The source is [qrcode-ai-com/qrcode-barcode-scanner-rust](https://github.com/qrcode-ai-com/qrcode-barcode-scanner-rust), and this notebook is
[`examples/kaggle/`](https://github.com/qrcode-ai-com/qrcode-barcode-scanner-rust/tree/main/examples/kaggle) in it. The scanner engine 0.11.0 is
distributed under AGPL-3.0-or-later and arrives through `pip install`; no engine source is pasted
in these cells. The cells themselves are MIT source in that repository, and Kaggle shows a public
notebook under the Apache 2.0 licence.

Choose **Run All**. Each cell checks its own result, so a cell that finishes without an error did
what it says. The last cell removes the package again and proves it is gone.

## 1. Install the package

The package has no dependency. The session needs Internet access, which is on for this notebook.

In [ ]:
%pip install --quiet qrcode-barcode-scanner==0.11.0

## 2. Check what is installed

The cell prints the Python version, the platform and the package version. It then compares the
SHA-256 of the installed native module with the modules of the wheels published on PyPI for this
version, so the output names the exact wheel this session runs.

In [ ]:
import hashlib
import importlib.metadata
import pathlib
import platform

import qrcode_barcode_scanner as scanner

VERSION = "0.11.0"
# SHA-256 of the native module inside each wheel published on PyPI for this version.
PUBLISHED_MODULES = {
    "326431ff9f8402df257c38bb2d26037ae03e73bb6a0e7a381c4f3c4c9ab5ed09":
        "qrcode_barcode_scanner-0.11.0-cp38-abi3-macosx_10_12_x86_64.whl",
    "d1975a7b933c7440b4bd45295946e9a082286a0d6d18df7bdfdfc53fdceffcc7":
        "qrcode_barcode_scanner-0.11.0-cp38-abi3-macosx_11_0_arm64.whl",
    "3e353686899d4658f5b5560392ca4ef377789b83a0ee4fa13b36eaa6f349145d":
        "qrcode_barcode_scanner-0.11.0-cp38-abi3-manylinux_2_17_aarch64.manylinux2014_aarch64.whl",
    "de30ac77aee57c90ec4552af10e075fbd83aa1f775b53031cf5acf84fc117e34":
        "qrcode_barcode_scanner-0.11.0-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl",
    "80bf6cde56de0f73954554093e68c98708a141021797b76f3217a81ba0072173":
        "qrcode_barcode_scanner-0.11.0-cp38-abi3-musllinux_1_2_x86_64.whl",
    "256537c2c1083a39901c4ea3ba509eed0617fc194258186c647a67ef554d3b2b":
        "qrcode_barcode_scanner-0.11.0-cp38-abi3-win_amd64.whl",
}

installed = importlib.metadata.version("qrcode-barcode-scanner")
assert installed == VERSION, f"version {installed} is installed, this notebook expects {VERSION}"
module = next(path for path in pathlib.Path(scanner.__file__).parent.iterdir()
              if path.suffix in (".so", ".pyd"))
digest = hashlib.sha256(module.read_bytes()).hexdigest()
wheel = PUBLISHED_MODULES.get(digest)

print("Python", platform.python_version(), "on", platform.system(), platform.machine())
print("qrcode-barcode-scanner", installed)
print("Native module SHA-256:", digest)
assert wheel is not None, "this native module is not one of the modules published on PyPI"
print("It is the module of the PyPI wheel", wheel)

## 3. Get the sample image

The sample holds one QR code and one Code 128 barcode. It comes from the scanner's repository at
the release tag, and the cell checks its SHA-256 before using it.

In [ ]:
import urllib.request

from IPython.display import Image, display

SAMPLE_URL = ("https://raw.githubusercontent.com/qrcode-ai-com/qrcode-barcode-scanner-rust/"
              "v0.11.0/fixtures/symbology/mixed-code128-qr.png")
SAMPLE_SHA256 = "c636e8d56df483ff1ec06dbe9b5bcba226c579f8fc1c3d10c4ccf4e100762ec8"

with urllib.request.urlopen(SAMPLE_URL, timeout=60) as response:
    sample = response.read()
assert hashlib.sha256(sample).hexdigest() == SAMPLE_SHA256, "the sample is not the expected file"
print("Sample:", len(sample), "bytes, SHA-256", SAMPLE_SHA256)
display(Image(data=sample, width=480))

## 4. Scan it with the three profiles

`full` runs every stage and scores how well a QR code scans. `fast` skips the deep QR stages and
gives a reduced score. `frame` only decodes, as for camera frames, and gives no score.
`budget_ms=0` removes the time budget, so every scan runs to its end: the budget flag printed for
each profile must be `False`.

In [ ]:
EXPECTED = [("qr_code", "https://qrc-ai.com/76xMa"), ("code128", "STOCK-4827")]

reports = {}
for profile in ("fast", "frame", "full"):
    report = scanner.scan(sample, profile=profile, budget_ms=0)
    reports[profile] = report
    found = [(row["symbology"], row["content"]["text"]) for row in report["detections"]]
    assert sorted(found) == sorted(EXPECTED), (profile, found)
    assert report["trace"]["budget_exhausted"] is False, profile
    assert report["trace"]["engine_panics"] == 0, profile
    assert report["versions"]["scanner"] == VERSION, report["versions"]
    assert (report["score"] is None) == (profile == "frame"), profile
    print(f"{profile:<5}", " · ".join(f"{symbology}: {text}" for symbology, text in found),
          "· budget_exhausted:", report["trace"]["budget_exhausted"])

score = reports["full"]["score"]
print("Scannability of the first code:", score["value"], "out of 100,", score["grade"])

## 5. Read a report

A report is a `dict`. Each detection gives the symbology, the decoded content, the kind of payload
and where each physical copy of the code was measured in the image.

In [ ]:
for row in reports["full"]["detections"]:
    print(row["symbology"])
    print("  text       ", row["content"]["text"])
    print("  payload    ", row["payload"]["kind"])
    for occurrence in row["occurrences"]:
        geometry = occurrence["geometry"]
        points = ", ".join(f"({point['x']:.0f}, {point['y']:.0f})" for point in geometry["points"])
        print("  measured as", geometry["kind"], "at", points)

## 6. Save the three reports

The reports are written as JSON files to `scan-reports/` in the notebook's working folder. On
Kaggle that folder is `/kaggle/working`, so the files appear under the notebook's **Output**.

In [ ]:
import json

out = pathlib.Path("scan-reports")
out.mkdir(exist_ok=True)
for profile, report in reports.items():
    path = out / f"{profile}.json"
    path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
    print(path, path.stat().st_size, "bytes")

## 7. An image without a code, and a file that is not an image

An image with no code gives an empty list: a valid result, not an error. Bytes that are not an
image raise `ValueError` with the code `QRS-001`.

In [ ]:
import struct
import zlib


def white_png(width=64, height=64):
    """A plain white PNG, built here so that the cell needs no other package."""
    def chunk(kind, data):
        body = kind + data
        return struct.pack(">I", len(data)) + body + struct.pack(">I", zlib.crc32(body))
    row = b"\x00" + b"\xff" * (3 * width)
    return (b"\x89PNG\r\n\x1a\n"
            + chunk(b"IHDR", struct.pack(">IIBBBBB", width, height, 8, 2, 0, 0, 0))
            + chunk(b"IDAT", zlib.compress(row * height)) + chunk(b"IEND", b""))


empty = scanner.scan(white_png(), profile="full", budget_ms=0)
assert empty["detections"] == [] and empty["trace"]["budget_exhausted"] is False
print("White image: no code found, and the scan finished.")

try:
    scanner.scan(b"not an image", profile="full", budget_ms=0)
except ValueError as error:
    assert "QRS-001" in str(error), error
    print("Not an image:", error)
else:
    raise AssertionError("bytes that are not an image were accepted")

## 8. Remove the package and prove it is gone

A new Python interpreter of this session imports the package first, then the package is
uninstalled, and the same new interpreter must now fail to import it and to find its metadata.
This notebook's own interpreter keeps the module it already loaded until the session ends.

In [ ]:
import os
import subprocess
import sys

IMPORT = "import qrcode_barcode_scanner"
METADATA = "import importlib.metadata as m; m.version('qrcode-barcode-scanner')"


def fresh(statement):
    """Run one statement in a new interpreter of this session."""
    plain = dict(os.environ, PYTHON_COLORS="0", NO_COLOR="1")
    return subprocess.run([sys.executable, "-c", statement], capture_output=True, text=True, env=plain)


before = fresh(IMPORT)
assert before.returncode == 0, before.stderr
print("Before: a new interpreter imports qrcode_barcode_scanner.")

uninstall = subprocess.run([sys.executable, "-m", "pip", "uninstall", "--yes", "qrcode-barcode-scanner"],
                           capture_output=True, text=True)
assert uninstall.returncode == 0, uninstall.stderr
print(uninstall.stdout.strip().splitlines()[-1])

after = fresh(IMPORT)
assert after.returncode != 0 and "ModuleNotFoundError" in after.stderr, after.stderr
print("After:", after.stderr.strip().splitlines()[-1])
metadata = fresh(METADATA)
assert metadata.returncode != 0 and "PackageNotFoundError" in metadata.stderr, metadata.stderr
print("After:", metadata.stderr.strip().splitlines()[-1])

## Go further

- [Python API](https://github.com/qrcode-ai-com/qrcode-barcode-scanner-rust/blob/v0.11.0/docs/api/python.mdx)
- [The report, field by field](https://github.com/qrcode-ai-com/qrcode-barcode-scanner-rust/tree/v0.11.0/spec)
- [How the score is computed](https://github.com/qrcode-ai-com/qrcode-barcode-scanner-rust/blob/v0.11.0/docs/SCORING.md)
- [The package on PyPI](https://pypi.org/project/qrcode-barcode-scanner/0.11.0/)

To run it again after the last cell, run the install cell first.